# Load the data set

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv('../data/training.csv')

In [3]:
print(df.shape)

(640840, 10)


In [4]:
print(df.columns)

Index(['Unnamed: 0', 'store_ID', 'day_of_week', 'date', 'nb_customers_on_day',
       'open', 'promotion', 'state_holiday', 'school_holiday', 'sales'],
      dtype='str')


In [5]:
print(df.dtypes)

Unnamed: 0             int64
store_ID               int64
day_of_week            int64
date                     str
nb_customers_on_day    int64
open                   int64
promotion              int64
state_holiday            str
school_holiday         int64
sales                  int64
dtype: object


In [6]:
print(df.isna())

        Unnamed: 0  store_ID  day_of_week   date  nb_customers_on_day   open  \
0            False     False        False  False                False  False   
1            False     False        False  False                False  False   
2            False     False        False  False                False  False   
3            False     False        False  False                False  False   
4            False     False        False  False                False  False   
...            ...       ...          ...    ...                  ...    ...   
640835       False     False        False  False                False  False   
640836       False     False        False  False                False  False   
640837       False     False        False  False                False  False   
640838       False     False        False  False                False  False   
640839       False     False        False  False                False  False   

        promotion  state_holiday  schoo

In [7]:
df.head()

,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
0,425390,366,4,2013-04-18,517,1,0,0,0,4422
1,291687,394,6,2015-04-11,694,1,0,0,0,8297
2,411278,807,4,2013-08-29,970,1,1,0,0,9729
3,664714,802,2,2013-05-28,473,1,1,0,0,6513
4,540835,726,4,2013-10-10,1068,1,1,0,0,10882


In [8]:
print(df.isna().sum())

Unnamed: 0             0
store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
sales                  0
dtype: int64


In [9]:
print('state holiday:', df['state_holiday'].unique())
print('number of customers per day:', df['nb_customers_on_day'].unique())
print('open:', df['open'].unique())

state holiday: <StringArray>
['0', 'a', 'c', 'b']
Length: 4, dtype: str
number of customers per day: [ 517  694  970 ... 4630 3713 4003]
open: [1 0]


In [10]:
df[df['sales'] == 0]

,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
6,600327,659,7,2014-06-08,0,0,0,0,0,0
10,561067,273,7,2014-10-05,0,0,0,0,0,0
18,409022,767,7,2013-01-27,0,0,0,0,0,0
34,605423,534,7,2014-06-08,0,0,0,0,0,0
35,231682,514,7,2014-03-09,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
640807,381567,167,7,2014-07-27,0,0,0,0,0,0
640812,49811,870,3,2014-07-23,0,0,0,0,1,0
640814,556209,650,7,2014-03-23,0,0,0,0,0,0
640834,304137,329,7,2013-09-15,0,0,0,0,0,0


In [11]:
df[df['nb_customers_on_day'] == 0]

,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
6,600327,659,7,2014-06-08,0,0,0,0,0,0
10,561067,273,7,2014-10-05,0,0,0,0,0,0
18,409022,767,7,2013-01-27,0,0,0,0,0,0
34,605423,534,7,2014-06-08,0,0,0,0,0,0
35,231682,514,7,2014-03-09,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
640807,381567,167,7,2014-07-27,0,0,0,0,0,0
640812,49811,870,3,2014-07-23,0,0,0,0,1,0
640814,556209,650,7,2014-03-23,0,0,0,0,0,0
640834,304137,329,7,2013-09-15,0,0,0,0,0,0


# Find the anomalous row

In [12]:
anomaly = df[(df['sales'] == 0) & (df['nb_customers_on_day'] > 0)]
print(anomaly)

        Unnamed: 0  store_ID  day_of_week        date  nb_customers_on_day  \
352744      319242       948            4  2013-04-25                    5   

        open  promotion state_holiday  school_holiday  sales  
352744     1          1             0               0      0  


# Data Cleaning

### Drop the row identifier

In [13]:
df.drop(columns=['Unnamed: 0'], inplace=True)
print('Dropped row-ID column. New shape:', df.shape)

Dropped row-ID column. New shape: (640840, 9)


### Fix `date` data types, extract useful features, drop the original `date` column

In [14]:
df['date']       = pd.to_datetime(df['date'])
print(df['date'].dtype)

df['year']       = df['date'].dt.year         # captures year-over-year trends
df['month']      = df['date'].dt.month        # captures seasonality (Dec = high sales)
df['day']        = df['date'].dt.day          # captures month-end effects
df['week']       = df['date'].dt.isocalendar().week.astype(int)  # weekly patterns
df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)  # weekend flag
df = df.drop(columns=['date'])

print(df.head())

datetime64[us]
   store_ID  day_of_week  nb_customers_on_day  open  promotion state_holiday  \
0       366            4                  517     1          0             0   
1       394            6                  694     1          0             0   
2       807            4                  970     1          1             0   
3       802            2                  473     1          1             0   
4       726            4                 1068     1          1             0   

   school_holiday  sales  year  month  day  week  is_weekend  
0               0   4422  2013      4   18    16           0  
1               0   8297  2015      4   11    15           1  
2               0   9729  2013      8   29    35           0  
3               0   6513  2013      5   28    22           0  
4               0  10882  2013     10   10    41           0  


### for column `state_holiday` Creates a binary column for each category, drops holiday_0 to avoid the dummy variable trap (multicollinearity)

In [15]:
# Must cast to str BEFORE get_dummies to ensure consistent type
df['state_holiday'] = df['state_holiday'].astype(str)
df = pd.get_dummies(df, columns=['state_holiday'], prefix='holiday', drop_first=True)

# Convert boolean OHE columns to int (0/1) for model compatibility
for col in ['holiday_a', 'holiday_b', 'holiday_c']:
    if col in df.columns:
        df[col] = df[col].astype(int)

print(df.head())

   store_ID  day_of_week  nb_customers_on_day  open  promotion  \
0       366            4                  517     1          0   
1       394            6                  694     1          0   
2       807            4                  970     1          1   
3       802            2                  473     1          1   
4       726            4                 1068     1          1   

   school_holiday  sales  year  month  day  week  is_weekend  holiday_a  \
0               0   4422  2013      4   18    16           0          0   
1               0   8297  2015      4   11    15           1          0   
2               0   9729  2013      8   29    35           0          0   
3               0   6513  2013      5   28    22           0          0   
4               0  10882  2013     10   10    41           0          0   

   holiday_b  holiday_c  
0          0          0  
1          0          0  
2          0          0  
3          0          0  
4          0          

In [16]:
df[(df['open'] == 0) & (df['sales'] > 0)]

,store_ID,day_of_week,nb_customers_on_day,open,promotion,school_holiday,sales,year,month,day,week,is_weekend,holiday_a,holiday_b,holiday_c


In [17]:
df[(df['open'] == 1) & (df['sales'] == 0)]

,store_ID,day_of_week,nb_customers_on_day,open,promotion,school_holiday,sales,year,month,day,week,is_weekend,holiday_a,holiday_b,holiday_c
54379,339,3,0,1,0,0,0,2013,1,30,5,0,0,0,0
61554,1039,3,0,1,0,0,0,2013,7,10,28,0,0,0,0
63404,983,5,0,1,0,0,0,2014,1,17,3,0,0,0,0
82644,387,4,0,1,0,1,0,2014,7,24,30,0,0,0,0
117219,762,4,0,1,0,0,0,2013,1,17,3,0,0,0,0
137903,303,4,0,1,0,1,0,2014,7,24,30,0,0,0,0
163742,25,3,0,1,0,0,0,2014,2,12,7,0,0,0,0
180912,835,4,0,1,0,0,0,2014,9,11,37,0,0,0,0
188539,835,3,0,1,0,0,0,2014,9,10,37,0,0,0,0
209801,548,5,0,1,1,1,0,2014,9,5,36,0,0,0,0


In [18]:
# Keep only open stores: closed stores always have sales=0
# They do not teach the model anything about what drives sales
# We will handle closed stores separately at prediction time (force sales=0)
df = df[df['open'] == 1]
print(df.shape)

(532016, 15)


In [19]:
# Quantify the anomaly as a percentage
open_with_customers = df[(df['open'] == 1) & (df['nb_customers_on_day'] > 0)]
total     = len(open_with_customers)
anomalies = len(open_with_customers[open_with_customers['sales'] == 0])
pct       = (anomalies / total) * 100
print(f'Total open days with customers: {total}')
print(f'Days with sales = 0:            {anomalies}')
print(f'Anomaly percentage:             {pct:.4f}%')

Total open days with customers: 531986
Days with sales = 0:            1
Anomaly percentage:             0.0002%


# Model 1: Linear Regression

define the features and target

In [20]:
# X = all features (everything except what we want to predict)
# y = target variable (what we want to predict)
X = df.drop(columns=['sales'])
y = df['sales']

# Save feature columns: needed to align REAL_DATA to same structure as training
feature_columns = X.columns.tolist()
print('Features:', feature_columns)

Features: ['store_ID', 'day_of_week', 'nb_customers_on_day', 'open', 'promotion', 'school_holiday', 'year', 'month', 'day', 'week', 'is_weekend', 'holiday_a', 'holiday_b', 'holiday_c']


train and test split

In [21]:
# Split into training (75%) and validation (25%) sets
# random_state=42 ensures reproducibility (same split every run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')

Train size: 399012 | Test size: 133004


train linear regression

In [22]:
# LinearRegression finds the best-fit line through the data
# It learns the weight (coefficient) of each feature
lr = LinearRegression()
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


evaluate

In [23]:
# R2 Train      - how well the model learned the training data
# R2 Validation - expected performance on unseen data (our prediction)
# RMSE          - average error in euros (lower = better)
# R2 Difference - gap between train and validation (close to 0 = good generalisation)

y_train_pred_lr = lr.predict(X_train)
y_val_pred_lr   = lr.predict(X_test)

r2_train_lr      = r2_score(y_train, y_train_pred_lr)
r2_validation_lr = r2_score(y_test,  y_val_pred_lr)
rmse_lr          = np.sqrt(mean_squared_error(y_test, y_val_pred_lr))
r2_difference_lr = r2_train_lr - r2_validation_lr

print('=== Linear Regression Results ===')
print(f'R2 Train      : {r2_train_lr:.4f}  <- how well model learned training data')
print(f'R2 Validation : {r2_validation_lr:.4f}  <- expected performance on real data')
print(f'RMSE          : {rmse_lr:.2f}  <- average error in euros')
print(f'R2 Difference : {r2_difference_lr:.4f}  <- gap (closer to 0 = better generalisation)')

=== Linear Regression Results ===
R2 Train      : 0.7350  <- how well model learned training data
R2 Validation : 0.7366  <- expected performance on real data
RMSE          : 1593.77  <- average error in euros
R2 Difference : -0.0016  <- gap (closer to 0 = better generalisation)


# Real Data
Load REAL_DATA -> preprocess -> predict with all models -> export best

In [24]:
df_real = pd.read_csv('../data/REAL_DATA.csv')
print('Shape:', df_real.shape)
print('Columns:', df_real.columns.tolist())
print('Dtypes:', df_real.dtypes)
print('Missing:', df_real.isna().sum())
df_real.head()

Shape: (71205, 9)
Columns: ['index', 'store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'open', 'promotion', 'state_holiday', 'school_holiday']
Dtypes: index                  int64
store_ID               int64
day_of_week            int64
date                     str
nb_customers_on_day    int64
open                   int64
promotion              int64
state_holiday            str
school_holiday         int64
dtype: object
Missing: index                  0
store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
dtype: int64


,index,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday
0,272371,415,7,01/03/2015,0,0,0,0,0
1,558468,27,7,29/12/2013,0,0,0,0,0
2,76950,404,3,19/03/2014,657,1,1,0,0
3,77556,683,2,29/01/2013,862,1,0,0,0
4,456344,920,3,19/03/2014,591,1,1,0,0


In [25]:
print('state_holiday unique:', df_real['state_holiday'].unique())
print('open unique:         ', df_real['open'].unique())
print('Closed rows:         ', len(df_real[df_real['open'] == 0]))
print('Open rows:           ', len(df_real[df_real['open'] == 1]))

state_holiday unique: <StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: str
open unique:          [0 1]
Closed rows:          12100
Open rows:            59105


In [26]:
# Drop index column (same as Unnamed: 0 in training)
df_real = df_real.drop(columns=['index'])

# Fix date: REAL_DATA uses DD/MM/YYYY (different from training YYYY-MM-DD)
# dayfirst=True is critical: without it 01/03/2015 reads as January 3rd not March 1st
df_real['date']       = pd.to_datetime(df_real['date'], dayfirst=True)
df_real['year']       = df_real['date'].dt.year
df_real['month']      = df_real['date'].dt.month
df_real['day']        = df_real['date'].dt.day
df_real['week']       = df_real['date'].dt.isocalendar().week.astype(int)
df_real['is_weekend'] = df_real['date'].dt.dayofweek.isin([5, 6]).astype(int)
df_real = df_real.drop(columns=['date'])

# Encode state_holiday: must cast to str BEFORE get_dummies
df_real['state_holiday'] = df_real['state_holiday'].astype(str)
df_real = pd.get_dummies(df_real, columns=['state_holiday'], prefix='holiday', drop_first=True)

# Convert boolean OHE columns to int
for col in ['holiday_a', 'holiday_b', 'holiday_c']:
    if col in df_real.columns:
        df_real[col] = df_real[col].astype(int)

# Align columns to exactly match training features
df_real = df_real.reindex(columns=feature_columns, fill_value=0)
print('Preprocessing done. Shape:', df_real.shape)
df_real.head()

Preprocessing done. Shape: (71205, 14)


,store_ID,day_of_week,nb_customers_on_day,open,promotion,school_holiday,year,month,day,week,is_weekend,holiday_a,holiday_b,holiday_c
0,415,7,0,0,0,0,2015,3,1,9,1,0,0,0
1,27,7,0,0,0,0,2013,12,29,52,1,0,0,0
2,404,3,657,1,1,0,2014,3,19,12,0,0,0,0
3,683,2,862,1,0,0,2013,1,29,5,0,0,0,0
4,920,3,591,1,1,0,2014,3,19,12,0,0,0,0


## Predict with Linear Regression

In [27]:
# Use df_real[feature_columns] to ensure correct columns
y_pred_lr_real = lr.predict(df_real[feature_columns])
print(y_pred_lr_real)

[1776.36500563 1894.72851004 7025.5069479  ... 5684.35444162 6774.15306859
 7952.79393351]


# Model 2: XGBoost

define the features and target
*(reusing X and y defined in Model 1)*

train and test split
*(reusing X_train, X_test, y_train, y_test defined in Model 1)*

train XGBoost

In [28]:
# XGBoost: gradient boosting model, handles non-linear patterns better than Linear Regression
# n_estimators: number of trees | learning_rate: step size | max_depth: tree depth
xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    verbosity=0
)
xgb.fit(X_train, y_train)
print('Model trained.')

Model trained.


evaluate

In [29]:
# R2 Train      - how well the model learned the training data
# R2 Validation - expected performance on unseen data (our prediction)
# RMSE          - average error in euros (lower = better)
# R2 Difference - gap between train and validation (close to 0 = good generalisation)

y_train_pred_xgb = xgb.predict(X_train)
y_val_pred_xgb   = xgb.predict(X_test)

r2_train_xgb      = r2_score(y_train, y_train_pred_xgb)
r2_validation_xgb = r2_score(y_test,  y_val_pred_xgb)
rmse_xgb          = np.sqrt(mean_squared_error(y_test, y_val_pred_xgb))
r2_difference_xgb = r2_train_xgb - r2_validation_xgb

print('=== XGBoost Results ===')
print(f'R2 Train      : {r2_train_xgb:.4f}  <- how well model learned training data')
print(f'R2 Validation : {r2_validation_xgb:.4f}  <- expected performance on real data')
print(f'RMSE          : {rmse_xgb:.2f}  <- average error in euros')
print(f'R2 Difference : {r2_difference_xgb:.4f}  <- gap (closer to 0 = better generalisation)')

=== XGBoost Results ===
R2 Train      : 0.8456  <- how well model learned training data
R2 Validation : 0.8423  <- expected performance on real data
RMSE          : 1233.31  <- average error in euros
R2 Difference : 0.0033  <- gap (closer to 0 = better generalisation)


## Predict Real data with XGBoost

In [30]:
# Use df_real[feature_columns] to ensure correct columns: avoids ValueError
y_pred_xgb_real = xgb.predict(df_real[feature_columns])
print(y_pred_xgb_real)

[1982.2963 1219.5327 6522.937  ... 5414.1177 6422.7505 7638.096 ]


# Model 3: Random Forest

define the features and target
*(reusing X and y defined in Model 1)*

train and test split
*(reusing X_train, X_test, y_train, y_test defined in Model 1)*

train Random Forest

In [31]:
# Random Forest: ensemble of decision trees, robust to outliers and overfitting
# n_estimators: number of trees | max_depth: tree depth limit | n_jobs=-1: use all CPU cores
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print('Model trained.')

Model trained.


evaluate

In [32]:
# R2 Train      - how well the model learned the training data
# R2 Validation - expected performance on unseen data (our prediction)
# RMSE          - average error in euros (lower = better)
# R2 Difference - gap between train and validation (close to 0 = good generalisation)

y_train_pred_rf = rf.predict(X_train)
y_val_pred_rf   = rf.predict(X_test)

r2_train_rf      = r2_score(y_train, y_train_pred_rf)
r2_validation_rf = r2_score(y_test,  y_val_pred_rf)
rmse_rf          = np.sqrt(mean_squared_error(y_test, y_val_pred_rf))
r2_difference_rf = r2_train_rf - r2_validation_rf

print('=== Random Forest Results ===')
print(f'R2 Train      : {r2_train_rf:.4f}  <- how well model learned training data')
print(f'R2 Validation : {r2_validation_rf:.4f}  <- expected performance on real data')
print(f'RMSE          : {rmse_rf:.2f}  <- average error in euros')
print(f'R2 Difference : {r2_difference_rf:.4f}  <- gap (closer to 0 = better generalisation)')

=== Random Forest Results ===
R2 Train      : 0.8170  <- how well model learned training data
R2 Validation : 0.8108  <- expected performance on real data
RMSE          : 1350.68  <- average error in euros
R2 Difference : 0.0062  <- gap (closer to 0 = better generalisation)


## Predict Real data with Random Forest

In [33]:
# Use df_real[feature_columns] to ensure correct columns: avoids ValueError
y_pred_rf_real = rf.predict(df_real[feature_columns])
print(y_pred_rf_real)

[  17.36         42.76       6809.55127208 ... 5532.52950327 6638.2852982
 7747.19191489]


# Model Comparison

In [34]:
# Compare all 3 models side by side
# R2 Validation = expected performance on real data | RMSE = average error in euros
comparison = pd.DataFrame({
    'Model'        : ['Linear Regression', 'XGBoost', 'Random Forest'],
    'R2 Train'     : [r2_train_lr,      r2_train_xgb,      r2_train_rf],
    'R2 Validation': [r2_validation_lr, r2_validation_xgb, r2_validation_rf],
    'RMSE'         : [rmse_lr,          rmse_xgb,          rmse_rf],
    'R2 Difference': [r2_difference_lr, r2_difference_xgb, r2_difference_rf]
})
comparison = comparison.round(4).sort_values('R2 Validation', ascending=False).reset_index(drop=True)
print('=== Model Comparison (sorted by R2 Validation) ===')
print(comparison.to_string(index=False))
print(f'\nBest model: {comparison.iloc[0]["Model"]}')

=== Model Comparison (sorted by R2 Validation) ===
            Model  R2 Train  R2 Validation      RMSE  R2 Difference
          XGBoost    0.8456         0.8423 1233.3117         0.0033
    Random Forest    0.8170         0.8108 1350.6834         0.0062
Linear Regression    0.7350         0.7366 1593.7695        -0.0016

Best model: XGBoost


# Export Final Predictions as group_4.csv

In [35]:
# Pick predictions from the best model automatically
best_model = comparison.iloc[0]['Model']
print(f'Best model: {best_model}')

if best_model == 'XGBoost':
    best_preds = y_pred_xgb_real
elif best_model == 'Random Forest':
    best_preds = y_pred_rf_real
else:
    best_preds = y_pred_lr_real

# Load original file to preserve the original format required by the teacher
df_output = pd.read_csv('../data/REAL_DATA.csv')
df_output['sales'] = np.clip(best_preds, 0, None).round().astype(int)

# Force closed stores = 0 (business rule: no customers = no sales)
df_output.loc[df_output['open'] == 0, 'sales'] = 0

df_output.to_csv('../data/group_4.csv', index=False)
print('Exported group_4.csv')

# Sanity check
print('Closed stores with sales > 0:', len(df_output[(df_output['open'] == 0) & (df_output['sales'] > 0)]), '<- must be 0')
print('\nPrediction stats (open stores):')
print(df_output[df_output['open'] == 1]['sales'].describe())

Best model: XGBoost
Exported group_4.csv
Closed stores with sales > 0: 0 <- must be 0

Prediction stats (open stores):
count    59105.000000
mean      6952.742898
std       2793.208913
min       1135.000000
25%       5096.000000
50%       6459.000000
75%       8196.000000
max      27808.000000
Name: sales, dtype: float64
